In [ ]:
"""
Portfolio optimization with linear multi-factor risk floors and exact cardinality.

Solves the following MILP using SCIP (via pyscipopt):

    maximize    sum_i (return_i - cost_i) * x_i
    subject to  sum_i R[i, k] * x_i >= const_vector[k]   for each risk metric k
                x_i >= lb_i * y_i                         for each asset i
                x_i <= ub_i * y_i                         for each asset i
                sum_i y_i == n                            (exact cardinality)
                lb_i <= x_i <= ub_i                       for each asset i
                y_i in {0, 1}                             for each asset i
"""

from __future__ import annotations

import pandas as pd
from pyscipopt import Model, quicksum # pyright: ignore[reportMissingImports]


def optimize_portfolio(
    risk_matrix: pd.DataFrame,
    cost_vector: pd.Series,
    return_vector: pd.Series,
    const_vector: pd.Series,
    n: int,
    bound: pd.DataFrame,
    verbose: bool = False,
) -> dict:
    """
    Solve the cardinality-constrained portfolio problem with linear risk floors.

    Parameters
    ----------
    risk_matrix : pd.DataFrame, shape (N, m)
        Rows indexed by asset, columns indexed by risk-metric name.
        Entry R[i, k] is asset i's exposure to risk metric k.
    cost_vector : pd.Series, length N
        Indexed by asset (must match risk_matrix.index).
    return_vector : pd.Series, length N
        Indexed by asset (must match risk_matrix.index).
    const_vector : pd.Series, length m
        Indexed by risk-metric name (must match risk_matrix.columns).
        Per-metric LOWER bound on aggregate portfolio exposure.
    n : int
        Exact number of assets to select (1 <= n <= N).
    bound : pd.DataFrame
        Indexed by asset (must match risk_matrix.index), with columns "lb" and "ub".
        "lb" is the per-asset minimum weight (may be negative for short positions).
        "ub" is the per-asset maximum weight.
        Must satisfy lb_i < ub_i for every asset i.
    verbose : bool, default False
        If True, SCIP prints solver progress.

    Returns
    -------
    dict with keys:
        "status"   : SCIP termination status (e.g., "optimal", "infeasible")
        "objective": objective value (None if not optimal)
        "weights"  : pd.Series of x_i values, indexed by asset
        "selected" : pd.Series of y_i values (0/1), indexed by asset
    """
    # ---- Input validation -------------------------------------------------
    assets = risk_matrix.index
    metrics = risk_matrix.columns
    N = len(assets)

    if not cost_vector.index.equals(assets):
        raise ValueError("cost_vector.index must match risk_matrix.index.")
    if not return_vector.index.equals(assets):
        raise ValueError("return_vector.index must match risk_matrix.index.")
    if not const_vector.index.equals(metrics):
        raise ValueError("const_vector.index must match risk_matrix.columns.")
    if not isinstance(n, int) or n < 1 or n > N:
        raise ValueError(f"n must be an integer in [1, {N}], got {n!r}.")
    if not isinstance(bound, pd.DataFrame) or not {"lb", "ub"}.issubset(bound.columns):
        raise ValueError("bound must be a DataFrame with columns 'lb' and 'ub'.")
    if not bound.index.equals(assets):
        raise ValueError("bound.index must match risk_matrix.index.")
    if (bound["lb"] >= bound["ub"]).any():
        raise ValueError("bound requires lb < ub for every asset.")

    # ---- Build the SCIP model --------------------------------------------
    model = Model("cardinality_portfolio")
    if not verbose:
        model.hideOutput()

    # Continuous weights x_i in [lb_i, ub_i]
    x = {
        i: model.addVar(name=f"x_{i}", vtype="C", lb=float(bound.at[i, "lb"]), ub=float(bound.at[i, "ub"]))
        for i in assets
    }
    # Binary selection indicators y_i
    y = {
        i: model.addVar(name=f"y_{i}", vtype="B")
        for i in assets
    }

    # Objective: maximize sum_i (return_i - cost_i) * x_i
    net = return_vector - cost_vector
    model.setObjective(
        quicksum(float(net[i]) * x[i] for i in assets),
        sense="maximize",
    )

    # Risk-floor constraints: one per risk metric column
    for k in metrics:
        model.addCons(
            quicksum(float(risk_matrix.at[i, k]) * x[i] for i in assets)
            >= float(const_vector[k]),
            name=f"risk_floor_{k}",
        )

    # Linking constraints: force x_i = 0 when y_i = 0, allow [lb_i, ub_i] when y_i = 1
    for i in assets:
        model.addCons(x[i] >= float(bound.at[i, "lb"]) * y[i], name=f"link_lb_{i}")
        model.addCons(x[i] <= float(bound.at[i, "ub"]) * y[i], name=f"link_ub_{i}")

    # Exact cardinality: sum_i y_i == n
    model.addCons(quicksum(y[i] for i in assets) == n, name="cardinality")

    # ---- Solve ------------------------------------------------------------
    model.optimize()
    status = model.getStatus()

    if status == "optimal":
        obj = model.getObjVal()
        weights = pd.Series(
            {i: model.getVal(x[i]) for i in assets}, name="weight"
        )
        selected = pd.Series(
            {i: int(round(model.getVal(y[i]))) for i in assets}, name="selected"
        )
    else:
        obj = None
        weights = pd.Series(index=assets, dtype=float, name="weight")
        selected = pd.Series(index=assets, dtype=int, name="selected")

    return {
        "status": status,
        "objective": obj,
        "weights": weights,
        "selected": selected,
    }

In [ ]:
# ----------------------------------------------------------------------------
# Example / smoke test
# ----------------------------------------------------------------------------
if __name__ == "__main__":
    import numpy as np

    rng = np.random.default_rng(0)
    assets = [f"A{i}" for i in range(8)]
    metrics = ["scenario_a", "scenario_b", "scenario_c"]

    risk_matrix = pd.DataFrame(
        rng.uniform(0.1, 1.0, size=(len(assets), len(metrics))),
        index=assets,
        columns=metrics,
    )
    cost_vector = pd.Series(rng.uniform(0.01, 0.05, size=len(assets)), index=assets)
    return_vector = pd.Series(rng.uniform(0.05, 0.25, size=len(assets)), index=assets)
    const_vector = pd.Series([0.5, 0.5, 0.5], index=metrics)
    bound = pd.DataFrame(
        {
            "lb": rng.uniform(-0.2, 0.0, size=len(assets)),
            "ub": rng.uniform(0.5, 1.0, size=len(assets)),
        },
        index=assets,
    )

In [ ]:
result = optimize_portfolio(
    risk_matrix=risk_matrix,
    cost_vector=cost_vector,
    return_vector=return_vector,
    const_vector=const_vector,
    n=3,
    bound=bound,
    verbose=False,
)

print(f"Status     : {result['status']}")
if result["objective"] is not None:
    print(f"Objective  : {result['objective']:.6f}")
else:
    print("Objective  : None")
print("\nWeights:")
print(result["weights"].round(4))
print("\nSelected:")
print(result["selected"])
print(f"\nNumber selected: {result['selected'].sum()}")